### Chapter 2.5 - Automatic Differentiation

Automatic differentiation is the system PyTorch uses to compute derivatives of tensor computations.

The main idea is simple: while computing outputs, PyTorch records how those outputs were created. Later, `backward()` walks through that recorded computation and applies the chain rule.


In [ ]:
import torch


### 2.5.1 A Simple Function

#### 1. Intuition

A computation graph records operations and dependencies between values.

A leaf tensor is a tensor that you create directly, rather than one produced by an operation. If a leaf tensor has `requires_grad=True`, PyTorch can store gradients for it.

#### 2. Why this exists

Manual derivative calculations become impossible to manage for large neural networks. Automatic differentiation handles the bookkeeping while still following the same calculus rules.


#### 3. Examples

Manual derivative for `y = 2*x*x`.


In [ ]:
x_value = 3.0
y_value = 2 * x_value * x_value
manual_grad = 4 * x_value
y_value, manual_grad


PyTorch automatic differentiation for the same function.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x * x
y.backward()
x.grad


Clear gradients before another backward pass. In training, this is usually done every update step.


In [ ]:
x.grad.zero_()
y = x + x
z = y * y
z.backward()
x.grad


#### 4. Step-by-step breakdown

`requires_grad=True` tells PyTorch to track operations that use `x`.

`y = 2 * x * x` creates a new tensor and records that it came from multiplication operations involving `x`.

`y.backward()` computes the derivative of `y` with respect to tracked leaf tensors.

`x.grad` stores the derivative value.

`x.grad.zero_()` changes the existing gradient tensor to zero in place. The underscore means the method modifies the object directly.

Gradients accumulate by default. Accumulate means new gradient values are added to existing gradient values instead of replacing them automatically.

#### 5. Connection to ML systems

During training, the model computes a loss, calls `backward()`, and then an optimizer uses gradients to update parameters.

An optimizer is an algorithm that changes parameters to reduce loss.

#### 6. Common confusion points

- `requires_grad=True` records future operations; it does not compute gradients immediately.
- `.backward()` starts gradient computation.
- `.grad` belongs to leaf tensors that require gradients.
- Gradients accumulate unless you clear them.


### 2.5.2 Backward for Non-Scalar Variables

#### 1. Intuition

A scalar has one value. A vector has multiple values.

By default, PyTorch expects `.backward()` to start from a scalar output. If the output has multiple values, we must either reduce it to a scalar or provide external gradient information.

#### 2. Why this exists

Loss functions usually reduce many per-example losses into one scalar because one scalar gives a single objective to optimize.


#### 3. Examples

Create a vector output, then reduce it with `sum` before calling backward.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x
loss = y.sum()
loss.backward()
x.grad


Use `mean` instead of `sum` to create a scalar objective.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x
loss = y.mean()
loss.backward()
x.grad


#### 4. Step-by-step breakdown

`x * x` creates a vector because `x` is a vector.

`y.sum()` reduces all vector entries into one scalar.

`loss.backward()` computes how that scalar loss changes with respect to each entry of `x`.

`y.mean()` also creates a scalar, but it averages values instead of adding them.

Because `mean` divides by the number of elements, its gradients are smaller than the gradients from `sum`.

#### 5. Connection to ML systems

Training commonly computes one loss per example, then averages the losses across a batch. This creates a scalar loss suitable for `backward()`.

#### 6. Common confusion points

- `.backward()` is simplest when the output is scalar.
- `sum` and `mean` both reduce, but they scale gradients differently.
- A vector output has multiple sensitivities, so PyTorch needs a clear scalar objective or supplied gradient.
- Reducing too early can hide per-example behavior during debugging.


### 2.5.3 Detaching Computation

#### 1. Intuition

Detaching means creating a tensor view of a value that is no longer connected to the computation graph.

The detached value keeps the numerical data but stops gradient tracking through that path.

#### 2. Why this exists

Sometimes we want to use a computed value as a constant. Detaching tells PyTorch: use this number, but do not let gradients flow backward through how it was computed.


#### 3. Examples

Compare a normal path with a detached path.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
u = y.detach()
z = u * x
z.backward()
x.grad


Without detaching, both paths contribute to the gradient.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
z = y * x
z.backward()
x.grad


#### 4. Step-by-step breakdown

In the first example, `y = x * x` is tracked.

`u = y.detach()` creates a value with the same number as `y` but disconnects it from the graph.

`z = u * x` still depends on `x` directly, but not through the computation that made `u`.

In the second example, `z = y * x` depends on `x` both through `y` and directly through `x`, so the gradient includes both paths.

#### 5. Connection to ML systems

Detach appears in advanced training patterns, logging, target networks, and cases where a model output should be reused without affecting gradients.

A target network is a separate model used to provide stable target values in some algorithms.

#### 6. Common confusion points

- Detach does not change the number.
- Detach changes gradient connectivity.
- Detached tensors can still be used in ordinary computation.
- Detaching accidentally can prevent a model from learning.


### 2.5.4 Gradients and Python Control Flow

#### 1. Intuition

Python control flow means ordinary Python decisions and loops, such as `if`, `for`, and `while`.

PyTorch records the operations that actually run. This means the computation graph can depend on Python control flow.

#### 2. Why this exists

Real model code is not always one straight formula. It may contain loops, conditionals, or shape-dependent behavior. Dynamic graph construction lets PyTorch differentiate through the path that was executed.


#### 3. Examples

A loop creates repeated operations that PyTorch can track.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x
for _ in range(3):
    y = y * 2

y.backward()
x.grad


A branch records only the path that actually runs.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
if x.item() > 0:
    y = x * x
else:
    y = -x

y.backward()
x.grad


#### 4. Step-by-step breakdown

The loop starts with `y = x`.

Each iteration replaces `y` with `y * 2`. After three iterations, the final value is `8 * x`.

PyTorch records the three multiplications that actually happened.

`x.item()` extracts a plain Python number so Python can decide which branch to run.

Only the selected branch contributes operations to the graph.

#### 5. Connection to ML systems

This dynamic behavior is useful for research code, where models may make data-dependent choices. It also means debugging requires knowing which Python path actually ran.

#### 6. Common confusion points

- PyTorch tracks tensor operations, not your intention.
- Python branches can change the graph from one run to another.
- `.item()` converts a one-value tensor into a Python number and leaves the tensor graph world.
- Loops are differentiable only through the tensor operations executed inside them.


### 2.5.5 Discussion

#### 1. Intuition

Automatic differentiation combines tensor computation with calculus bookkeeping.

You write the forward computation. PyTorch records the graph. Calling `backward()` applies the chain rule backward through the graph.

#### 2. Why this exists

This separation keeps model code readable. You define how predictions and losses are computed, while the framework handles derivative mechanics.


#### 3. Examples

A minimal training-style gradient step without an optimizer object.


In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 1) ** 2
loss.backward()
with torch.no_grad():
    w -= 0.1 * w.grad
w


#### 4. Step-by-step breakdown

`loss = (w - 1) ** 2` creates a scalar loss. The loss is lowest when `w` equals 1.

`loss.backward()` computes the derivative of the loss with respect to `w`.

`with torch.no_grad():` temporarily tells PyTorch not to track operations inside the block.

The update `w -= 0.1 * w.grad` changes the parameter using its gradient.

The `with` statement is Python syntax for running a block of code under a temporary context. Here, the context disables gradient tracking for the update.

#### 5. Connection to ML systems

Optimizers such as stochastic gradient descent automate this parameter update pattern. Later training loops will repeat forward pass, loss computation, backward pass, and parameter update many times.

#### 6. Common confusion points

- Updating parameters should not usually be tracked as part of the gradient graph.
- `torch.no_grad()` is used for operations that should not build derivative history.
- Autograd computes gradients; it does not choose the training objective for you.
- Shape and scalar-loss issues still matter even with automatic differentiation.


### 2.5.6 Exercises

#### 1. Intuition

These exercises practice reading and controlling gradient flow.

#### 2. Why this exists

Automatic differentiation feels less mysterious when you can predict gradients for small examples.


#### 3. Examples

Exercise 1: Compute the gradient of `y = 3*x^2` at `x = 2`.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x * x
y.backward()
x.grad


Exercise 2: Compare `sum` and `mean` reductions.


In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * x
loss = y.mean()
loss.backward()
x.grad


Exercise 3: Detach an intermediate value and observe the remaining path.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x * 2
z = y.detach() * x
z.backward()
x.grad


Exercise 4: Clear gradients before a second backward pass.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
z = x * x
z.backward()
x.grad.zero_()
q = x * x
q.backward()
x.grad


#### 4. Step-by-step breakdown

Exercise 1 checks a simple polynomial derivative.

Exercise 2 checks reduction scaling.

Exercise 3 checks detach behavior.

Exercise 4 checks gradient clearing.

#### 5. Connection to ML systems

These are the same mechanics used inside training loops, only without a full model class yet.

#### 6. Common confusion points

- Reusing a tensor with an old `.grad` value can confuse debugging.
- Detach affects gradient paths, not numerical values.
- Reductions change gradient scale.
- Use small examples when autograd behavior is unclear.
